# TCS-Prompt Inference Notebook

This notebook contains the TCS-Prompt model implementation and an inference-only evaluation workflow.

In [1]:
import os
import sys
import math
import random
import warnings
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.nn.parameter import Parameter
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

warnings.filterwarnings("ignore", category=UserWarning)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / "outputs").exists() else NOTEBOOK_DIR.parent

candidate_checkpoints = sorted(REPO_ROOT.glob("outputs/**/best_model_acc0.9250_ep75.pth"))
default_checkpoint = candidate_checkpoints[0] if candidate_checkpoints else Path("checkpoints/tcs_prompt_best_model.pth")
CHECKPOINT_PATH = Path(os.environ.get("TCS_PROMPT_CHECKPOINT", str(default_checkpoint))).expanduser()

CODE_BACKUP_ROOT = CHECKPOINT_PATH.parent / "code_backup"
if CODE_BACKUP_ROOT.exists():
    sys.path.insert(0, str(CODE_BACKUP_ROOT))
else:
    sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = None
if os.environ.get("FER_DATA_ROOT"):
    DATA_ROOT = Path(os.environ["FER_DATA_ROOT"]).expanduser()
else:
    for ancestor in [REPO_ROOT, *REPO_ROOT.parents]:
        candidate = ancestor / "fer_dataset"
        if candidate.exists():
            DATA_ROOT = candidate
            break
if DATA_ROOT is None:
    DATA_ROOT = Path("path/to/fer_dataset")

if os.environ.get("CLIP_MODEL_PATH"):
    CLIP_MODEL_PATH = Path(os.environ["CLIP_MODEL_PATH"]).expanduser()
else:
    clip_candidates = [Path.home() / ".cache" / "clip" / "ViT-B-16.pt", REPO_ROOT / "ViT-B-16.pt"]
    CLIP_MODEL_PATH = next((p for p in clip_candidates if p.exists()), Path("path/to/ViT-B-16.pt"))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ["surprise", "fear", "disgusted", "happy", "sad", "anger", "neutral"]


## Dataset and Evaluation Utilities

The test-time preprocessing follows the archived experiment code. Labels are mapped to the RAF-DB seven-class order used by the classifier head.

In [2]:
class FacialExpressionDataset(Dataset):
    classes = CLASS_NAMES
    label_mapping = {
        "RAFDB": {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6},
        "SFEW": {0: 6, 1: 5, 2: 0, 3: 2, 4: 1, 5: 3, 6: 4},
        "FER2013": {0: 6, 1: 5, 2: 0, 3: 2, 4: 1, 5: 3, 6: 4},
        "AffectNet": {0: 6, 1: 3, 2: 4, 3: 0, 4: 1, 5: 2, 6: 5},
        "MMA": {0: 5, 1: 2, 2: 1, 3: 3, 4: 6, 5: 4, 6: 0},
        "FERplus": {"Anger": 5, "Disgust": 2, "Fear": 1, "Happy": 3, "Neutral": 6, "Sadness": 4, "Surprise": 0},
    }

    def __init__(self, root, domain, split="test"):
        self.root = Path(root).expanduser()
        self.domain = domain
        self.split = split
        interpolation = InterpolationMode.BILINEAR if domain == "RAFDB" else InterpolationMode.BICUBIC
        self.transform = transforms.Compose([
            transforms.Resize((224, 224), interpolation=interpolation),
            lambda image: image.convert("RGB"),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        self.data, self.targets = self._load_items()

    def _load_items(self):
        label_file = self.root / self.domain / "labels" / f"{self.split}.txt"
        image_root = self.root / self.domain / "images" / self.split
        if not label_file.exists():
            raise FileNotFoundError(f"Missing label file for {self.domain}/{self.split}. Set FER_DATA_ROOT to the dataset root.")

        mapping = self.label_mapping.get(self.domain)
        data, targets = [], []
        with label_file.open("r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 2:
                    continue
                rel_path, raw_label = parts[0], parts[1]
                if mapping is None:
                    label = int(raw_label)
                elif raw_label in mapping:
                    label = mapping[raw_label]
                else:
                    raw_int = int(raw_label)
                    if raw_int not in mapping:
                        continue
                    label = mapping[raw_int]
                data.append(image_root / rel_path)
                targets.append(label)
        return data, targets

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        image_path = self.data[index]
        label = self.targets[index]
        try:
            image = Image.open(image_path).convert("RGB")
            image = self.transform(image)
        except Exception:
            image = torch.zeros((3, 224, 224))
        return {"img": image, "label": label, "img0": image}


def make_loader(domain="RAFDB", split=None, batch_size=16, workers=4):
    split = split or ("val" if domain == "FERplus" else "test")
    dataset = FacialExpressionDataset(DATA_ROOT, domain=domain, split=split)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=torch.cuda.is_available())
    return dataset, loader


class AverageMeter:
    def __init__(self):
        self.total = 0.0
        self.count = 0

    def update(self, value, n):
        self.total += float(value) * int(n)
        self.count += int(n)

    @property
    def value(self):
        return self.total / max(self.count, 1)


## Complete TCS-Prompt Model Code

In [3]:
from clip import clip
from clip.simple_tokenizer import SimpleTokenizer

_TOKENIZER = SimpleTokenizer()


def enable_attention_capture(block):
    def custom_forward(self, x: torch.Tensor):
        qkv = self.ln_1(x)
        attn_out, attn_weights = self.attn(
            qkv,
            qkv,
            qkv,
            need_weights=True,
            average_attn_weights=False,
            attn_mask=self.attn_mask.to(dtype=x.dtype, device=x.device) if self.attn_mask is not None else None,
        )
        self.current_self_attn = attn_weights
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return x

    import types
    block.forward = types.MethodType(custom_forward, block)


def enable_attention_capture_for_clip(clip_model):
    for block in clip_model.visual.transformer.resblocks:
        enable_attention_capture(block)
    for block in clip_model.transformer.resblocks:
        enable_attention_capture(block)


class AdaptiveCausalIntervention(nn.Module):
    def __init__(self, noise_sigma=0.1, noise_max_prob=0.5, noise_min_prob=0.05):
        super().__init__()
        self.noise_sigma = noise_sigma
        self.noise_max_prob = noise_max_prob
        self.noise_min_prob = noise_min_prob

    @staticmethod
    def normalize_importance(scores):
        min_vals = torch.min(scores, dim=1, keepdim=True)[0]
        max_vals = torch.max(scores, dim=1, keepdim=True)[0]
        ranges = torch.clamp(max_vals - min_vals, min=1e-12)
        return (scores - min_vals) / ranges

    def importance_to_probability(self, normalized_score):
        return self.noise_max_prob - normalized_score * (self.noise_max_prob - self.noise_min_prob)

    def forward(self, hidden_states, token_importance_score, is_vision, target_mask1, target_mask2):
        if self.noise_sigma <= 0 or self.noise_max_prob < self.noise_min_prob:
            return hidden_states

        batch_size, seq_len, _ = hidden_states.shape
        _, score_len = token_importance_score.shape
        modified_hidden_states = hidden_states.clone()
        normalized_importance = self.normalize_importance(token_importance_score)
        full_scores = torch.full(
            (batch_size, seq_len), 0.5, device=hidden_states.device, dtype=token_importance_score.dtype
        )

        if is_vision:
            if seq_len == score_len + 1:
                full_scores[:, 1:] = normalized_importance
            elif seq_len == score_len:
                full_scores = normalized_importance
            else:
                raise ValueError("Vision score length does not match hidden-state length.")
        else:
            if seq_len == score_len + 1:
                full_scores[:, :-1] = normalized_importance
            elif seq_len == score_len:
                full_scores = normalized_importance
            else:
                raise ValueError("Text score length does not match hidden-state length.")

        target_mask1 = target_mask1.to(hidden_states.device)
        target_mask2 = target_mask2.to(hidden_states.device)
        if target_mask1.dim() == 1:
            target_mask1 = target_mask1.unsqueeze(0).expand(batch_size, -1)
        if target_mask2.dim() == 1:
            target_mask2 = target_mask2.unsqueeze(0).expand(batch_size, -1)
        target_mask = target_mask1 | target_mask2
        gaussian_noise = torch.randn_like(hidden_states) * self.noise_sigma + 1.0

        for i in range(batch_size):
            target_indices = torch.where(target_mask[i])[0]
            if target_indices.numel() == 0:
                continue
            probabilities = self.importance_to_probability(full_scores[i, target_indices])
            selected = target_indices[torch.rand(target_indices.numel(), device=hidden_states.device) < probabilities]
            if selected.numel() > 0:
                modified_hidden_states[i, selected, :] *= gaussian_noise[i, selected, :]
        return modified_hidden_states


class CrossModalSemanticDecoupling(nn.Module):
    def __init__(self, vis_dim=768, text_dim=512, base_proj=None):
        super().__init__()
        self.proj = nn.Linear(vis_dim, text_dim, bias=False)
        if base_proj is not None and base_proj.t().shape == self.proj.weight.shape:
            with torch.no_grad():
                self.proj.weight.copy_(base_proj.t().clone().detach())
        else:
            nn.init.orthogonal_(self.proj.weight)
        self.temp = 0.1

    def forward(self, mid_patches, au_text_features):
        proj_patches = self.proj(mid_patches)
        norm_patches = F.normalize(proj_patches, p=2, dim=-1)
        norm_texts = F.normalize(au_text_features, p=2, dim=-1)
        au_sim = norm_patches @ norm_texts.t()
        au_sim = au_sim.view(mid_patches.shape[0], -1, 12, 2)
        bipolar_score = au_sim[:, :, :, 0] - au_sim[:, :, :, 1]
        patch_scores = bipolar_score.max(dim=-1)[0]
        au_attn = F.softmax(patch_scores / self.temp, dim=-1)
        with torch.no_grad():
            stats = {
                "max": au_attn.max(dim=-1)[0].mean().item(),
                "entropy": (-torch.sum(au_attn * torch.log(au_attn + 1e-9), dim=-1)).mean().item(),
                "top5": torch.topk(au_attn, k=5, dim=-1)[0].sum(dim=-1).mean().item(),
            }
        return proj_patches, au_attn, stats


class HypergraphConvolution(nn.Module):
    def __init__(self, in_ft, out_ft, bias=True):
        super().__init__()
        self.weight = Parameter(torch.Tensor(in_ft, out_ft))
        if bias:
            self.bias = Parameter(torch.Tensor(out_ft))
        else:
            self.register_parameter("bias", None)
        self.reset_parameters()

    def reset_parameters(self):
        stdv = 1.0 / math.sqrt(self.weight.size(1))
        self.weight.data.uniform_(-stdv, stdv)
        if self.bias is not None:
            self.bias.data.uniform_(-stdv, stdv)

    def forward(self, x, G):
        x = x.matmul(self.weight)
        if self.bias is not None:
            x = x + self.bias
        return G.matmul(x)


class HypergraphNetwork(nn.Module):
    def __init__(self, in_ch, n_hid, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.hgc1 = HypergraphConvolution(in_ch, n_hid)
        self.hgc2 = HypergraphConvolution(n_hid, n_hid)

    def forward(self, x, G):
        x = F.relu(self.hgc1(x, G))
        x = F.dropout(x, self.dropout, training=self.training)
        return self.hgc2(x, G)


def build_masked_hypergraph(x, k_neig=4, valid_mask=None):
    batch_size, max_nodes, _ = x.shape
    dist_mat = torch.cdist(x, x, p=2.0)
    if valid_mask is not None:
        invalid = ~valid_mask
        dist_mat = dist_mat.masked_fill(invalid.unsqueeze(1), 1e9)
        dist_mat = dist_mat.masked_fill(invalid.unsqueeze(2), 1e9)

    _, knn_idx = torch.topk(dist_mat, k=min(k_neig, max_nodes), dim=-1, largest=False)
    H = torch.zeros(batch_size, max_nodes, max_nodes, device=x.device, dtype=x.dtype)
    H.scatter_(2, knn_idx, 1.0)
    if valid_mask is not None:
        invalid = ~valid_mask
        H = H.masked_fill(invalid.unsqueeze(1), 0.0)
        H = H.masked_fill(invalid.unsqueeze(2), 0.0)

    DV = H.sum(dim=2).clamp(min=1e-6)
    DE = H.sum(dim=1).clamp(min=1e-6)
    H_invDE = H * torch.pow(DE, -1.0).unsqueeze(-1).transpose(1, 2)
    G = torch.bmm(H_invDE, H.transpose(1, 2))
    DV2 = torch.pow(DV, -0.5).unsqueeze(-1)
    return G * DV2 * DV2.transpose(1, 2)


class HighOrderTopologicalModeling(nn.Module):
    def __init__(self, feat_dim=768, k_neig=4, num_cycle=1, min_nodes=5, max_nodes=50):
        super().__init__()
        self.k_neig = k_neig
        self.num_cycle = num_cycle
        self.min_nodes = min_nodes
        self.max_nodes = max_nodes
        self.max_entropy = math.log(196)
        self.HGNN = HypergraphNetwork(feat_dim, feat_dim, dropout=0.1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, mid_patches, au_attn):
        batch_size, num_patches, _ = mid_patches.shape
        entropies = -torch.sum(au_attn * torch.log(au_attn + 1e-9), dim=-1)
        dynamic_ratios = 1.0 - (entropies / self.max_entropy)
        dynamic_ks = torch.clamp((num_patches * dynamic_ratios).long(), min=self.min_nodes, max=self.max_nodes)
        max_k = dynamic_ks.max().item()
        _, topk_idx_all = torch.topk(au_attn, k=max_k, dim=-1)
        batch_idx = torch.arange(batch_size, device=mid_patches.device).unsqueeze(-1).expand(-1, max_k)
        x_topk = mid_patches[batch_idx, topk_idx_all, :]
        valid_mask = torch.arange(max_k, device=mid_patches.device).unsqueeze(0).expand(batch_size, -1) < dynamic_ks.unsqueeze(-1)

        feat_topk = x_topk
        for _ in range(self.num_cycle):
            G = build_masked_hypergraph(feat_topk, k_neig=self.k_neig, valid_mask=valid_mask)
            feat_topk = self.HGNN(feat_topk, G)
        feat_topk_updated = self.gamma * feat_topk * valid_mask.unsqueeze(-1)

        updated = torch.zeros_like(mid_patches)
        updated[batch_idx, topk_idx_all, :] = feat_topk_updated
        return mid_patches + updated


class PromptTextEncoder(nn.Module):
    def __init__(self, clip_model):
        super().__init__()
        self.transformer = clip_model.transformer
        self.positional_embedding = clip_model.positional_embedding
        self.ln_final = clip_model.ln_final
        self.text_projection = clip_model.text_projection
        self.dtype = clip_model.dtype

    def forward(self, prompts, tokenized_prompts, compound_prompts_deeper_text, return_attention=False, is_train=False):
        x = prompts + self.positional_embedding.type(self.dtype)
        x = x.permute(1, 0, 2)
        eos_indices = tokenized_prompts.argmax(dim=-1)
        batch_size = x.shape[1]
        all_self_attns, all_eos_attns = [], []

        for i, block in enumerate(self.transformer.resblocks):
            n_ctx = 0
            if i > 0 and (i - 1) < len(compound_prompts_deeper_text):
                deep_prompt = compound_prompts_deeper_text[i - 1].unsqueeze(1).expand(-1, x.shape[1], -1)
                n_ctx = deep_prompt.shape[0]
                x[1:1 + n_ctx, :, :] = deep_prompt
            x = block(x)
            if is_train and i <= len(compound_prompts_deeper_text) and return_attention:
                self_attn = block.current_self_attn
                batch_indices = torch.arange(batch_size, device=self_attn.device)
                eos_attn = self_attn[batch_indices, :, eos_indices, :].mean(dim=1)[:, :-1]
                all_self_attns.append(self_attn)
                all_eos_attns.append(eos_attn)

        x = x.permute(1, 0, 2)
        x = self.ln_final(x).type(self.dtype)
        x = x[torch.arange(x.shape[0]), eos_indices] @ self.text_projection
        if return_attention:
            return x, all_self_attns, all_eos_attns
        return x


class TCSPromptLearner(nn.Module):
    def __init__(self, cfg, classnames, clip_model):
        super().__init__()
        n_cls = len(classnames)
        n_ctx = cfg.TRAINER.TCS.N_CTX
        ctx_init = cfg.TRAINER.TCS.CTX_INIT
        dtype = clip_model.dtype
        ctx_dim = clip_model.ln_final.weight.shape[0]
        self.compound_prompts_depth = cfg.TRAINER.TCS.PROMPT_DEPTH

        if ctx_init and n_ctx <= 4:
            prompt = clip.tokenize(ctx_init.replace("_", " ")).to(clip_model.token_embedding.weight.device)
            with torch.no_grad():
                embedding = clip_model.token_embedding(prompt).type(dtype)
            ctx_vectors = embedding[0, 1:1 + n_ctx, :]
            prompt_prefix = ctx_init
        else:
            ctx_vectors = torch.empty(n_ctx, ctx_dim, dtype=dtype)
            nn.init.normal_(ctx_vectors, std=0.02)
            prompt_prefix = " ".join(["X"] * n_ctx)

        self.ctx = nn.Parameter(ctx_vectors)
        self.compound_prompts_text = nn.ParameterList([
            nn.Parameter(torch.empty(n_ctx, 512)) for _ in range(self.compound_prompts_depth - 1)
        ])
        for param in self.compound_prompts_text:
            nn.init.normal_(param, std=0.02)

        self.v_n_ctx = getattr(cfg.TRAINER.TCS, "V_N_CTX", n_ctx)
        v_ctx_dim = clip_model.visual.ln_pre.weight.shape[0]
        self.v_ctx = nn.Parameter(torch.empty(self.v_n_ctx, v_ctx_dim, dtype=dtype))
        nn.init.normal_(self.v_ctx, std=0.02)
        self.compound_prompts_vision = nn.ParameterList([
            nn.Parameter(torch.empty(self.v_n_ctx, v_ctx_dim, dtype=dtype)) for _ in range(self.compound_prompts_depth - 1)
        ])
        for param in self.compound_prompts_vision:
            nn.init.normal_(param, std=0.02)

        classnames = [name.replace("_", " ") for name in classnames]
        self.name_lens = [len(_TOKENIZER.encode(name)) for name in classnames]
        prompts = [prompt_prefix + " a photo of a " + name + " facial expression, a type of facial expression." for name in classnames]
        tokenized_prompts = torch.cat([clip.tokenize(p) for p in prompts]).to(clip_model.token_embedding.weight.device)
        with torch.no_grad():
            embedding = clip_model.token_embedding(tokenized_prompts).type(dtype)
        self.register_buffer("token_prefix", embedding[:, :1, :])
        self.register_buffer("token_suffix", embedding[:, 1 + n_ctx:, :])
        self.n_cls = n_cls
        self.n_ctx = n_ctx
        self.tokenized_prompts = tokenized_prompts

    def construct_prompts(self, ctx, prefix, suffix, label=None):
        if label is not None:
            prefix = prefix[label]
            suffix = suffix[label]
        return torch.cat([prefix, ctx, suffix], dim=1)

    def forward(self):
        ctx = self.ctx
        if ctx.dim() == 2:
            ctx = ctx.unsqueeze(0).expand(self.n_cls, -1, -1)
        prompts = self.construct_prompts(ctx, self.token_prefix, self.token_suffix)
        return prompts, self.v_ctx, self.compound_prompts_text, self.compound_prompts_vision


class TCSPromptModel(nn.Module):
    def __init__(self, args, classnames, clip_model):
        super().__init__()
        enable_attention_capture_for_clip(clip_model)
        self.prompt_generator = TCSPromptLearner(args, classnames, clip_model)
        self.tokenized_prompts = self.prompt_generator.tokenized_prompts
        self.image_encoder = clip_model.visual
        self.text_encoder = PromptTextEncoder(clip_model)
        self.logit_scale = clip_model.logit_scale
        self.dtype = clip_model.dtype
        self.use_csd = bool(getattr(args, "use_csd", True))
        self.use_aci = bool(getattr(args, "use_aci", True)) and self.use_csd
        self.use_htm = bool(getattr(args, "use_htm", True)) and self.use_csd
        self.aci = AdaptiveCausalIntervention()

        device = clip_model.token_embedding.weight.device
        frozen_prompts = ["a photo of a " + name.replace("_", " ") + " facial expression, a type of facial expression." for name in classnames]
        frozen_tokenized_prompts = torch.cat([clip.tokenize(p) for p in frozen_prompts]).to(device)
        with torch.no_grad():
            frozen_text_features = clip_model.encode_text(frozen_tokenized_prompts)
            frozen_text_features = frozen_text_features / frozen_text_features.norm(dim=-1, keepdim=True)
        self.register_buffer("frozen_text_features", frozen_text_features)

        au_descriptions = [
            "Middle part of eyebrows raised, indicating surprise or concern.", "Eyebrows level or slightly lowered, suggesting calmness or disapproval.",
            "Outer edges of eyebrows lifted, indicating curiosity or skepticism.", "Eyebrows flat or slightly lowered, suggesting indifference or certainty.",
            "Outer eyebrows lifted, indicating curiosity or skepticism.", "Outer eyebrows lowered or relaxed, signaling acceptance or neutrality.",
            "Upper eyelids lifted, signaling alertness or heightened attention.", "Upper eyelids lowered or drooping, signaling tiredness or disengagement.",
            "Cheek muscles lift upward, creating a warm or friendly smile.", "Cheeks unmoved or pulled downward, reflecting sadness or anger.",
            "Nose scrunched and pinched, indicating disgust or suspicion.", "Nose smooth and relaxed, indicating openness or neutrality.",
            "Corners of mouth pulled upward, signaling happiness or a smile.", "Corners of mouth pulled downward or neutral, signaling sadness or detachment.",
            "Corners of mouth pulled downward, indicating sadness or disapproval.", "Corners of mouth raised upward, indicating happiness or approval.",
            "Chin raised, signaling confidence or defiance with head tilted back.", "Chin lowered toward chest, signaling submission or insecurity.",
            "Mouth sides stretched outward, exposing teeth, signaling surprise or fear.", "Mouth closed or slightly pursed, signaling control or restraint.",
            "Visible mouth opening, signaling beginning of speech or surprise.", "Mouth tightly closed, signaling silence or emotional suppression.",
            "Lower face drops suddenly, showing shock or astonishment.", "Lower face composed and steady, showing calmness or emotional control.",
        ]
        au_tokenized = torch.cat([clip.tokenize(p) for p in au_descriptions]).to(device)
        with torch.no_grad():
            au_text_features = clip_model.encode_text(au_tokenized)
            au_text_features = au_text_features / au_text_features.norm(dim=-1, keepdim=True)
        self.register_buffer("au_text_features", au_text_features)

        vis_dim = clip_model.visual.ln_pre.weight.shape[0]
        native_proj = clip_model.visual.proj
        self.csd_modules = nn.ModuleDict({str(layer): CrossModalSemanticDecoupling(vis_dim=vis_dim, text_dim=512, base_proj=native_proj) for layer in range(6, 12)})
        self.htm_modules = nn.ModuleDict({str(layer): HighOrderTopologicalModeling(feat_dim=vis_dim, k_neig=4, num_cycle=1) for layer in range(6, 11)})

    def encode_visual_branch(self, image, v_ctx, deep_prompts_vision, return_attention=False, is_train=False):
        x = self.image_encoder.conv1(image.type(self.dtype))
        x = x.reshape(x.shape[0], x.shape[1], -1).permute(0, 2, 1)
        cls = self.image_encoder.class_embedding.to(x.dtype) + torch.zeros(x.shape[0], 1, x.shape[-1], dtype=x.dtype, device=x.device)
        x = torch.cat([cls, x], dim=1)
        x = x + self.image_encoder.positional_embedding.to(x.dtype)
        x = self.image_encoder.ln_pre(x)

        v_ctx = v_ctx.unsqueeze(0).expand(x.shape[0], -1, -1)
        x = torch.cat([x[:, :1, :], v_ctx, x[:, 1:, :]], dim=1)
        v_n_ctx = v_ctx.shape[1]
        all_self_attns, all_cls_attns, aux_pooled_features, layer_stats = [], [], [], {}
        x = x.permute(1, 0, 2)

        for layer_idx, block in enumerate(self.image_encoder.transformer.resblocks):
            if layer_idx > 0 and (layer_idx - 1) < len(deep_prompts_vision):
                deep_prompt = deep_prompts_vision[layer_idx - 1].unsqueeze(1).expand(-1, x.shape[1], -1)
                x[1:1 + v_n_ctx, :, :] = deep_prompt
            x = block(x)

            if return_attention or (self.use_csd and str(layer_idx) in self.csd_modules):
                self_attn = block.current_self_attn
                if return_attention:
                    all_self_attns.append(self_attn)
                    all_cls_attns.append(self_attn[:, :, 0, 1:].mean(dim=1))

                if self.use_csd and str(layer_idx) in self.csd_modules:
                    x_nld = x.permute(1, 0, 2)
                    mid_patches = x_nld[:, -196:, :]
                    proj_patches, au_attn, stats = self.csd_modules[str(layer_idx)](mid_patches, self.au_text_features)
                    stats["au_attn"] = au_attn.detach()
                    layer_stats[f"L{layer_idx}"] = stats
                    aux_pooled_features.append((proj_patches * au_attn.unsqueeze(-1)).sum(dim=1))

                    if self.use_htm and str(layer_idx) in self.htm_modules:
                        mid_patches = self.htm_modules[str(layer_idx)](mid_patches, au_attn)
                        x_nld = torch.cat([x_nld[:, :-196, :], mid_patches], dim=1)

                    token_importance = torch.cat([torch.zeros(x_nld.shape[0], v_n_ctx, device=x.device), au_attn], dim=1) if v_n_ctx > 0 else au_attn
                    if is_train and self.use_aci:
                        seq_len = x.shape[0]
                        target_mask1 = torch.zeros(seq_len, dtype=torch.bool, device=x.device)
                        target_mask2 = torch.zeros(seq_len, dtype=torch.bool, device=x.device)
                        target_mask1[1 + v_n_ctx:] = True
                        x_nld = self.aci(x_nld, token_importance, is_vision=True, target_mask1=target_mask1, target_mask2=target_mask2)
                    x = x_nld.permute(1, 0, 2)

        x = x.permute(1, 0, 2)
        x = self.image_encoder.ln_post(x[:, 0, :])
        if self.image_encoder.proj is not None:
            x = x @ self.image_encoder.proj
        if return_attention:
            return x, all_self_attns, all_cls_attns, aux_pooled_features, layer_stats
        return x, aux_pooled_features, layer_stats

    def forward(self, image, label=None, img_o=None, return_stats=False):
        tokenized_prompts = self.tokenized_prompts
        logit_scale = self.logit_scale.exp()
        is_train = self.prompt_generator.training
        prompts, v_ctx, deep_text, deep_vision = self.prompt_generator()

        text_outputs = self.text_encoder(prompts, tokenized_prompts, deep_text, return_attention=True, is_train=is_train)
        text_features = text_outputs[0] if isinstance(text_outputs, tuple) else text_outputs
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        if is_train and self.use_aci:
            with torch.no_grad():
                clean_outputs = self.encode_visual_branch(image, v_ctx, deep_vision, return_attention=False, is_train=False)
                image_features_clean = clean_outputs[0] if isinstance(clean_outputs, tuple) else clean_outputs
                image_features_clean = image_features_clean / image_features_clean.norm(dim=-1, keepdim=True)
                logits_clean = logit_scale * image_features_clean @ text_features.t()
            vision_outputs = self.encode_visual_branch(image, v_ctx, deep_vision, return_attention=True, is_train=True)
        elif is_train:
            vision_outputs = self.encode_visual_branch(image, v_ctx, deep_vision, return_attention=True, is_train=False)
            logits_clean = None
        else:
            vision_outputs = self.encode_visual_branch(image, v_ctx, deep_vision, return_attention=True, is_train=False)

        image_features = vision_outputs[0]
        aux_pooled_features = vision_outputs[3] if len(vision_outputs) > 3 else []
        layer_stats = vision_outputs[4] if len(vision_outputs) > 4 else {}
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        logits = logit_scale * image_features @ text_features.t()

        if is_train:
            with torch.no_grad():
                teacher_input = img_o if img_o is not None else image
                zs_out = self.image_encoder(teacher_input.type(self.dtype))
                zs_image_embedd = zs_out[1] if isinstance(zs_out, tuple) else zs_out
                zs_image_embedd = zs_image_embedd / zs_image_embedd.norm(dim=-1, keepdim=True)
            tt_logits = logit_scale * text_features @ self.frozen_text_features.t()
            tt_logits_inverse = logit_scale * self.frozen_text_features @ text_features.t()
            loss_aux = torch.tensor(0.0, device=image.device)
            if aux_pooled_features and label is not None:
                for v_pool in aux_pooled_features:
                    v_pool = F.normalize(v_pool, p=2, dim=-1)
                    loss_aux += F.cross_entropy(logit_scale * v_pool @ text_features.t(), label)
                loss_aux = loss_aux / len(aux_pooled_features)
            loss_consistency = F.kl_div(
                F.log_softmax(logits, dim=-1), F.softmax(logits_clean.detach(), dim=-1), reduction="batchmean"
            ) if self.use_aci else torch.tensor(0.0, device=image.device)
            return logits, image_features, zs_image_embedd, tt_logits, tt_logits_inverse, loss_aux, loss_consistency, layer_stats

        if return_stats:
            return logits, layer_stats
        return logits


def make_tcs_config(n_ctx=6, v_n_ctx=6, prompt_depth=9, cross_layer=6, use_csd=True, use_aci=True, use_htm=True):
    tcs = SimpleNamespace(N_CTX=n_ctx, V_N_CTX=v_n_ctx, CTX_INIT=False, PROMPT_DEPTH=prompt_depth, CROSS_LAYER=cross_layer)
    trainer = SimpleNamespace(TCS=tcs)
    return SimpleNamespace(TRAINER=trainer, use_csd=use_csd, use_aci=use_aci, use_htm=use_htm)


def translate_archived_checkpoint_keys(state_dict):
    translated = {}
    for key, value in state_dict.items():
        key = key.replace("prompt_learner.", "prompt_generator.")
        key = key.replace("aligners.", "csd_modules.")
        key = key.replace("hypergraphs.", "htm_modules.")
        translated[key] = value
    return translated


def load_tcs_prompt_model(checkpoint_path=CHECKPOINT_PATH, clip_model_path=CLIP_MODEL_PATH, device=DEVICE):
    if not Path(checkpoint_path).exists():
        raise FileNotFoundError("Checkpoint not found. Set TCS_PROMPT_CHECKPOINT or place the checkpoint under the expected repository output directory.")
    if not Path(clip_model_path).exists():
        raise FileNotFoundError("CLIP ViT-B/16 weights not found. Set CLIP_MODEL_PATH to the local ViT-B-16.pt file.")
    clip_model, _ = clip.load(str(clip_model_path), device=device, jit=False)
    clip_model.float()
    args = make_tcs_config()
    model = TCSPromptModel(args, CLASS_NAMES, clip_model).to(device)
    checkpoint = torch.load(str(checkpoint_path), map_location=device)
    status = model.load_state_dict(translate_archived_checkpoint_keys(checkpoint), strict=True)
    model.eval()
    return model, status


## Load Best Weights and Run Inference

The following cell loads the best archived weights, builds the model, and evaluates the RAF-DB test split. Set `FER_DATA_ROOT`, `CLIP_MODEL_PATH`, or `TCS_PROMPT_CHECKPOINT` if the files are stored outside the repository-relative defaults.

In [4]:
@torch.no_grad()
def evaluate(model, loader, device=DEVICE):
    criterion = nn.CrossEntropyLoss()
    loss_meter = AverageMeter()
    correct = 0
    total = 0
    for batch in loader:
        images = batch["img"].to(device, non_blocking=True)
        labels = torch.as_tensor(batch["label"], device=device).long()
        logits = model(images)
        loss = criterion(logits, labels)
        loss_meter.update(loss.item(), labels.shape[0])
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.numel()
    return {"samples": total, "loss": loss_meter.value, "accuracy": correct / max(total, 1)}


def run_domain_inference(model, domain, split=None, batch_size=16, workers=4):
    split = split or ("val" if domain == "FERplus" else "test")
    dataset, loader = make_loader(domain=domain, split=split, batch_size=batch_size, workers=workers)
    metrics = evaluate(model, loader)
    return {
        "model": "TCS-Prompt",
        "domain": domain,
        "test_loss": f"{metrics['loss']:.4f}",
        "test_accuracy": f"{metrics['accuracy']:.4f}",
        "checkpoint_epoch": 75,
    }


def run_all_domain_inference(domains=("RAFDB", "SFEW", "AffectNet", "FERplus"), batch_size=16, workers=4):
    model, load_status = load_tcs_prompt_model()
    return [
        run_domain_inference(model, domain=domain, batch_size=batch_size, workers=workers)
        for domain in domains
    ]


In [5]:
results = run_all_domain_inference(batch_size=16, workers=4)
results

[{'model': 'TCS-Prompt',
  'domain': 'RAFDB',
  'test_loss': '0.2913',
  'test_accuracy': '0.9250',
  'checkpoint_epoch': 75},
 {'model': 'TCS-Prompt',
  'domain': 'SFEW',
  'test_loss': '2.6149',
  'test_accuracy': '0.5367',
  'checkpoint_epoch': 75},
 {'model': 'TCS-Prompt',
  'domain': 'AffectNet',
  'test_loss': '3.2226',
  'test_accuracy': '0.5021',
  'checkpoint_epoch': 75},
 {'model': 'TCS-Prompt',
  'domain': 'FERplus',
  'test_loss': '0.8420',
  'test_accuracy': '0.8035',
  'checkpoint_epoch': 75}]